# Session 08 · Lab — Building Classifiers with Generative Models
*Desarrollo y Evaluación de Modelos Propios · Universidad del Rosario · Rosario GSB*

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/08-generative-classifiers/lab.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/08-generative-classifiers/lab.ipynb)

**Scenario.** *Andes Bank* (a fictional digital bank) wants to tag financial news tweets as **bearish**, **bullish** or **neutral** so its market-intelligence desk can react to bad news about the companies its clients hold. We will build four very different classifiers for the same task and compare them on quality, calibration, latency and cost.

**Learning objectives**
1. Train a generative classifier (Naive Bayes) and a discriminative one (logistic regression) and read their learning curves (Ng & Jordan, 2002).
2. Train an MLP on sentence embeddings with a manual **PyTorch** training loop, and see the same model in Keras.
3. Use an LLM as a **zero-shot generative classifier** by scoring the log-likelihood of each label.
4. Run a **synthetic data augmentation** experiment for a rare class, without leaking into the test set.
5. Summarise accuracy, macro-F1, calibration (ECE), latency and an estimated cost per 1,000 predictions.

| Part | Topic | Time |
|---|---|---|
| 0 | Data: financial tweets (with Part 1) | — |
| 1 | Naive Bayes vs logistic regression + learning curve | 18' |
| 2 | PyTorch MLP on MiniLM embeddings (+ Keras) | 12' |
| 3 | Zero-shot by label log-likelihood with Qwen2.5-0.5B-Instruct | 15' |
| 4 | Synthetic augmentation of the rare class | 10' |
| 5 | Summary table and decision | 5' |

**Kaggle checklist**
- Settings → **Internet on** (the dataset and the models are downloaded from Hugging Face).
- Settings → Accelerator → **GPU T4 x2** recommended. CPU works too: Parts 3 and 4 are just slower.
- Run the cells in order with Shift + Enter. ✋ **Checkpoints** are answered in the Zoom chat; 🧪 **Try it** exercises are optional stretch goals with a hidden solution.

In [ ]:
# --- Course setup (do not edit) ---
import os, sys, random, subprocess, importlib
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"  # used by the course smoke test
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

def ensure(package, import_name=None):
    """Install a package only if it is missing (Kaggle already ships most of them)."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Kaggle: {IN_KAGGLE} | Colab: {IN_COLAB} | Fast dev run: {FAST_DEV_RUN}")
# --- end of course setup ---
ensure("datasets")
ensure("transformers")
ensure("scikit-learn", "sklearn")
ensure("sentence-transformers", "sentence_transformers")

## Part 0 — Data: financial news tweets

We use [`zeroshot/twitter-financial-news-sentiment`](https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment): English tweets about markets and companies, each labelled **0 = bearish** (bad news for the stock), **1 = bullish** (good news) or **2 = neutral**. It has a `train` split (9,543 tweets) and a `validation` split (2,388 tweets). We use `validation` as our **test set** and never train or tune on it.

The next cell sets every size in the notebook. `FAST_DEV_RUN` is only used by the course's automatic test; in class it is `False` and you run the full sizes.

In [ ]:
# Sizes for the whole notebook (the automatic course test uses the small values)
N_TRAIN_MAX = 1500 if FAST_DEV_RUN else None      # None = the full training split
N_TEST_MAX = 400 if FAST_DEV_RUN else None        # None = the full validation split (our test set)
LC_SIZES = [20, 80, 320, None] if FAST_DEV_RUN else [20, 40, 80, 160, 320, 640, 1280, 2560, 5000, None]
LC_REPEATS = 1 if FAST_DEV_RUN else 3             # random training samples per size
MLP_EPOCHS = 3 if FAST_DEV_RUN else 25
N_LLM_TEST = 12 if FAST_DEV_RUN else 150          # test tweets scored by the LLM (slow part)
N_LLM_DEV = 6 if FAST_DEV_RUN else 30             # labelled training tweets to choose the scoring rule
N_SMALL = 120                                     # tiny training set for the augmentation experiment
N_SYNTH = 4 if FAST_DEV_RUN else 30               # synthetic tweets to generate
GEN_MAX_NEW_TOKENS = 24 if FAST_DEV_RUN else 48

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

LABELS = ["bearish", "bullish", "neutral"]        # index = label id in the dataset

raw = load_dataset("zeroshot/twitter-financial-news-sentiment")
train_df = raw["train"].shuffle(seed=SEED).to_pandas()   # the raw split is sorted by label: shuffle it
test_df = raw["validation"].to_pandas()
if N_TRAIN_MAX:
    train_df = train_df.iloc[:N_TRAIN_MAX].reset_index(drop=True)
if N_TEST_MAX:
    test_df = test_df.sample(n=N_TEST_MAX, random_state=SEED).reset_index(drop=True)

print(f"Train: {len(train_df):,} tweets | Test: {len(test_df):,} tweets")
dist = pd.DataFrame({
    "train": train_df.label.value_counts(normalize=True).sort_index(),
    "test": test_df.label.value_counts(normalize=True).sort_index(),
}).rename(index=dict(enumerate(LABELS)))
print((dist * 100).round(1).astype(str) + " %")
for k, name in enumerate(LABELS):
    print(f"\n{name.upper()} examples:")
    for t in train_df[train_df.label == k].text.head(2):
        print("  •", t)

We will compare every model with the same two metrics:
- **Accuracy**: share of correct predictions. With 65% neutral tweets, a model that always answers "neutral" already gets about 65%.
- **Macro-F1**: the average of the F1 score of each class, so the rare classes count as much as "neutral". This is our main metric.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def quality(y_true, y_pred):
    """Accuracy and macro-F1, rounded for display."""
    return {"accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4)}

always_neutral = np.full(len(test_df), 2)
print("Baseline 'always neutral':", quality(test_df.label, always_neutral))

## Part 1 — Naive Bayes vs logistic regression

Both models use the same features: **TF-IDF** weights of words and word pairs (bigrams).
- **Multinomial Naive Bayes** is *generative*: it models how each class produces words, $p(w \mid y)$, plus the class prior $p(y)$, and classifies with Bayes' rule. Training is just counting.
- **Logistic regression** is *discriminative*: it learns $p(y \mid x)$ directly by minimising cross-entropy with an optimiser.

⚠️ The vectorizer is fitted on the **training** texts only. Fitting it on all the data (including test) is a form of data leakage — you will hunt for it in the challenge.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

def make_vectorizer():
    return TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)

def make_nb():
    return MultinomialNB(alpha=0.1)                    # alpha = Laplace smoothing

def make_lr():
    return LogisticRegression(C=10, max_iter=2000)     # C = inverse of the L2 regularisation strength

vectorizer = make_vectorizer()
X_train = vectorizer.fit_transform(train_df.text)      # fit on training data only
X_test = vectorizer.transform(test_df.text)
nb = make_nb().fit(X_train, train_df.label)
lr = make_lr().fit(X_train, train_df.label)

print(f"Vocabulary: {len(vectorizer.vocabulary_):,} features (words and bigrams)")
print("Naive Bayes        :", quality(test_df.label, nb.predict(X_test)))
print("Logistic regression:", quality(test_df.label, lr.predict(X_test)))
print("\nLogistic regression, per class:")
print(classification_report(test_df.label, lr.predict(X_test), target_names=LABELS, digits=3))

### Naive Bayes is a (very crude) generative model of text
Because NB stores $p(w \mid y)$, we can read the most characteristic words of each class — and we can even **sample** a "tweet" from it. The result is word salad: NB ignores word order. An LLM is a far better generator, but the classification idea (Bayes' rule over a model of text) is the same.

In [ ]:
vocab = np.array(vectorizer.get_feature_names_out())
log_p = nb.feature_log_prob_                            # shape (3 classes, vocabulary)
for k, name in enumerate(LABELS):
    distinctive = log_p[k] - np.delete(log_p, k, axis=0).max(axis=0)   # how much more likely than in any other class
    top = np.argsort(distinctive)[::-1][:10]
    print(f"{name:>8}: {', '.join(vocab[top])}")

rng_np = np.random.default_rng(SEED)
p_bearish = np.exp(log_p[0]) / np.exp(log_p[0]).sum()
print("\nA 'bearish tweet' sampled from Naive Bayes:", " ".join(rng_np.choice(vocab, size=10, p=p_bearish)))

### Learning curve: who learns faster?
Ng & Jordan (2002) showed that Naive Bayes approaches its (usually higher) asymptotic error with fewer examples than logistic regression. We train both models on growing, **stratified** training samples (same class proportions as the full training set), several random samples per size, and evaluate on the same test set.

In [ ]:
def stratified_sample(df, n, seed):
    """n rows with the class proportions of df (at least 2 per class)."""
    rs = np.random.RandomState(seed)
    props = df.label.value_counts(normalize=True).sort_index()
    idx = []
    for c, p in props.items():
        k = max(2, int(round(n * p)))
        idx += list(rs.choice(np.where(df.label.values == c)[0], k, replace=False))
    return df.iloc[idx]

rows = []
t0 = time.time()
for n in LC_SIZES:
    repeats = LC_REPEATS if n else 1
    for r in range(repeats):
        sub = stratified_sample(train_df, n, seed=r) if n else train_df
        vec = make_vectorizer()
        A, B = vec.fit_transform(sub.text), vec.transform(test_df.text)
        for name, model in [("Naive Bayes", make_nb()), ("Logistic regression", make_lr())]:
            pred = model.fit(A, sub.label).predict(B)
            rows.append({"n_train": len(sub), "model": name, **quality(test_df.label, pred)})
curve = pd.DataFrame(rows).groupby(["n_train", "model"])[["accuracy", "macro_f1"]].mean().unstack("model")
print(f"Learning curve computed in {time.time() - t0:.1f} s")
print(curve.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["macro_f1", "accuracy"]):
    for name, style in [("Naive Bayes", "o-"), ("Logistic regression", "s--")]:
        ax.plot(curve.index, curve[(metric, name)], style, label=name)
    ax.set_xscale("log"); ax.set_xlabel("training tweets (log scale)"); ax.set_ylabel(metric)
    ax.grid(alpha=0.3); ax.legend()
axes[1].axhline((test_df.label == 2).mean(), color="gray", ls=":", label="always neutral")
axes[1].legend()
plt.suptitle("Generative (NB) vs discriminative (LR) learning curves")
plt.tight_layout(); plt.show()

✋ **Checkpoint (chat):** at roughly which training size does logistic regression overtake Naive Bayes on macro-F1? Why is accuracy a poor guide at the smallest sizes?

🧪 **Try it:** Naive Bayes is usually trained on raw word **counts**. Replace the TF-IDF vectorizer by `CountVectorizer(ngram_range=(1, 2))` for NB only and compare the full-data macro-F1.

<details><summary>Show a solution</summary>

```python
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(ngram_range=(1, 2))
nb_counts = MultinomialNB(alpha=0.1).fit(cv.fit_transform(train_df.text), train_df.label)
print(quality(test_df.label, nb_counts.predict(cv.transform(test_df.text))))
```
The difference is usually small; what matters is comparing both variants on the same test set.
</details>

## Part 2 — A PyTorch MLP on sentence embeddings

Instead of word counts, we represent each tweet with a **sentence embedding**: a 384-dimensional vector from the pretrained encoder `sentence-transformers/all-MiniLM-L6-v2` (Reimers & Gurevych, 2019). The encoder was trained without our labels — the same idea as using a VAE's latent code as features (Kingma et al., 2014, model M1). On top we train a small **MLP** (a logistic regression with one hidden layer) with an explicit PyTorch training loop.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

def embed(texts):
    return encoder.encode(list(texts), batch_size=64, convert_to_numpy=True, show_progress_bar=False)

t0 = time.time()
E_train_all = embed(train_df.text)
E_test = embed(test_df.text)
print(f"Embeddings: train {E_train_all.shape}, test {E_test.shape} in {time.time() - t0:.1f} s")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Hold out 10% of the training set as validation to monitor training (the test set stays untouched)
perm = np.random.RandomState(SEED).permutation(len(train_df))
n_val = len(perm) // 10
val_idx, tr_idx = perm[:n_val], perm[n_val:]
y_all = train_df.label.values

def to_tensor(a, dtype=torch.float32):
    return torch.tensor(a, dtype=dtype)

train_loader = DataLoader(TensorDataset(to_tensor(E_train_all[tr_idx]), to_tensor(y_all[tr_idx], torch.long)),
                          batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(SEED))
X_val_t, y_val = to_tensor(E_train_all[val_idx]).to(DEVICE), y_all[val_idx]

class MLP(nn.Module):
    def __init__(self, d_in=384, d_hidden=128, n_classes=3, p_drop=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Dropout(p_drop),
                                 nn.Linear(d_hidden, n_classes))

    def forward(self, x):
        return self.net(x)                               # logits; softmax lives inside the loss

torch.manual_seed(SEED)
mlp = MLP().to(DEVICE)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
history, best_f1, best_state = [], -1.0, None

for epoch in range(1, MLP_EPOCHS + 1):
    mlp.train()
    total, count = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()                            # 1. reset gradients
        loss = F.cross_entropy(mlp(xb), yb)              # 2. forward pass + loss
        loss.backward()                                  # 3. backpropagation
        optimizer.step()                                 # 4. update the weights
        total += loss.item() * len(yb); count += len(yb)
    mlp.eval()
    with torch.no_grad():
        val_pred = mlp(X_val_t).argmax(1).cpu().numpy()
    val_f1 = f1_score(y_val, val_pred, average="macro")
    history.append({"epoch": epoch, "train_loss": total / count, "val_macro_f1": val_f1})
    if val_f1 > best_f1:                                 # keep the best epoch (early stopping on validation)
        best_f1, best_state = val_f1, {k: v.detach().clone() for k, v in mlp.state_dict().items()}

mlp.load_state_dict(best_state)
hist = pd.DataFrame(history).set_index("epoch")
print(hist.round(4).tail())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
hist["train_loss"].plot(ax=axes[0], marker="o", title="training loss")
hist["val_macro_f1"].plot(ax=axes[1], marker="o", title="validation macro-F1")
plt.tight_layout(); plt.show()

mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(to_tensor(E_test).to(DEVICE)).argmax(1).cpu().numpy()
print("MLP on MiniLM embeddings (test):", quality(test_df.label, mlp_pred))

### The same model in Keras (optional)
The syllabus says "PyTorch **or** TensorFlow": here is the same MLP in Keras 3, which runs on TensorFlow (Kaggle and Colab default), JAX or PyTorch. `compile` + `fit` replace our hand-written loop. If Keras is not available in your environment, the cell just prints a message.

In [ ]:
try:
    import keras
    keras.utils.set_random_seed(SEED)
    k_model = keras.Sequential([
        keras.Input(shape=(384,)),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(3, activation="softmax"),
    ])
    k_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    k_model.fit(E_train_all[tr_idx], y_all[tr_idx], epochs=MLP_EPOCHS, batch_size=64,
                validation_data=(E_train_all[val_idx], y_val), verbose=0)
    k_pred = k_model.predict(E_test, verbose=0).argmax(1)
    print(f"Keras MLP (backend: {keras.backend.backend()}) (test):", quality(test_df.label, k_pred))
except Exception as exc:  # optional cell: never block the lab
    print("Keras cell skipped:", type(exc).__name__, exc)

✋ **Checkpoint (chat):** the MLP trained on about 8,600 of the 9,543 labelled tweets (90% of the training split; the rest was its validation set), while logistic regression in Part 1 used all of them. Did the pretrained embeddings help? What does the MLP need that NB and LR did not (think about the encoder)?

🧪 **Try it:** bearish is the rarest class. Pass class weights to the loss, `F.cross_entropy(logits, yb, weight=w)` with `w` inversely proportional to the class frequencies, retrain, and compare the bearish recall.

<details><summary>Show a solution</summary>

```python
counts = np.bincount(y_all[tr_idx], minlength=3)
w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float32, device=DEVICE)
# ...inside the training loop:
loss = F.cross_entropy(mlp(xb), yb, weight=w)
```
Recall of the rare classes usually goes up and their precision goes down: a trade-off you choose according to the cost of each error.
</details>

## Part 3 — Zero-shot classification by label log-likelihood

Now an LLM becomes a **generative classifier**. We never let it write free text. Instead, for each tweet $x$ and each label word (the **verbalizer** $v(y)$: `bearish`, `bullish`, `neutral`) we compute how likely the model finds that word as its answer:

$$ s(y) = \sum_{t=1}^{|v(y)|} \log p_\theta\big(v(y)_t \mid \text{prompt}(x),\ v(y)_{<t}\big) \qquad \hat y = \arg\max_y s(y) $$

- The prompt is built with the model's **chat template** (system + user message, then the assistant turn starts).
- We append each label's tokens after the prompt and run **one forward pass for the three labels** (a batch of 3, padded).
- The logits at position $t-1$ predict the token at position $t$: that off-by-one shift is the classic bug.
- A softmax over the three scores gives a confidence for each label, which we use for calibration in Part 5.

We use `Qwen/Qwen2.5-0.5B-Instruct`, small enough for a CPU. We convert it to float32: recent `transformers` versions load it in bfloat16, which is slow on CPUs and on the T4 GPU.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME).float().to(DEVICE).eval()
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

SYSTEM_PROMPT = "You are a financial analyst. You label the sentiment of financial news tweets."
VERBALIZERS = ["bearish", "bullish", "neutral"]          # same order as LABELS
LABEL_TOKEN_IDS = [tokenizer(v, add_special_tokens=False).input_ids for v in VERBALIZERS]

def build_messages(tweet):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Tweet: {tweet}\n\nIs this tweet bearish, bullish or neutral for the "
                                        "stock or market it mentions? Answer with one lowercase word: "
                                        "bearish, bullish or neutral."}]

def prompt_ids(tweet):
    enc = tokenizer.apply_chat_template(build_messages(tweet), add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True)
    return enc["input_ids"][0]

for v, ids in zip(VERBALIZERS, LABEL_TOKEN_IDS):
    print(f"{v!r:>10} -> tokens {tokenizer.convert_ids_to_tokens(ids)}")
print("\nWhat the model actually reads:\n")
print(tokenizer.decode(prompt_ids(test_df.text.iloc[0])))

In [ ]:
@torch.no_grad()
def label_scores(tweet):
    """Sum and mean of the log-probabilities of each verbalizer's tokens after the prompt."""
    p = prompt_ids(tweet)
    seqs = [torch.cat([p, torch.tensor(ids)]) for ids in LABEL_TOKEN_IDS]
    L = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), L), PAD_ID)
    attention = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):                          # right padding: real tokens keep their positions
        input_ids[i, :len(s)] = s
        attention[i, :len(s)] = 1
    logits = llm(input_ids=input_ids.to(DEVICE), attention_mask=attention.to(DEVICE)).logits
    sums, means = [], []
    for i, ids in enumerate(LABEL_TOKEN_IDS):
        pos = torch.arange(len(p) - 1, len(p) - 1 + len(ids), device=logits.device)   # shift by one
        target = torch.tensor(ids, device=logits.device)
        token_logp = logits[i, pos].float().log_softmax(-1).gather(1, target[:, None]).squeeze(1)
        sums.append(token_logp.sum().item())
        means.append(token_logp.mean().item())
    return sums, means

def score_texts(texts):
    out_sum, out_mean = [], []
    for t in texts:
        s, m = label_scores(t)
        out_sum.append(s); out_mean.append(m)
    return np.array(out_sum), np.array(out_mean)

s, m = label_scores(test_df.text.iloc[0])
print("Tweet:", test_df.text.iloc[0])
print("True label:", LABELS[test_df.label.iloc[0]])
print("Sum of log-probs :", dict(zip(VERBALIZERS, np.round(s, 3))))
print("Mean per token   :", dict(zip(VERBALIZERS, np.round(m, 3))))

### Sum or mean? Decide on a development set, never on the test set
`bearish` and `bullish` are two tokens each (`bear` + `ish`), `neutral` is one. The **sum** is the exact log-probability of the whole word. The **mean** divides by the number of tokens, a common heuristic for labels of different lengths — but once the model has written `bear`, the token `ish` is almost certain, so the mean halves the penalty of the two-token labels.

Which rule works better is an empirical question. We answer it on a small **development set** of labelled *training* tweets, then apply the chosen rule once to the test tweets.

In [ ]:
dev_df = train_df.sample(n=N_LLM_DEV, random_state=1)
llm_test_df = test_df.sample(n=N_LLM_TEST, random_state=SEED)

t0 = time.time()
dev_sum, dev_mean = score_texts(dev_df.text)
f1_sum = f1_score(dev_df.label, dev_sum.argmax(1), average="macro")
f1_mean = f1_score(dev_df.label, dev_mean.argmax(1), average="macro")
RULE = "sum" if f1_sum >= f1_mean else "mean"
print(f"Development set ({len(dev_df)} tweets): macro-F1 sum = {f1_sum:.3f} | mean = {f1_mean:.3f} -> rule = {RULE!r}")

test_sum, test_mean = score_texts(llm_test_df.text)
llm_seconds = time.time() - t0
llm_scores = test_sum if RULE == "sum" else test_mean
llm_pred = llm_scores.argmax(1)
llm_probs = torch.softmax(torch.tensor(llm_scores), dim=1).numpy()
llm_latency_ms = 1000 * llm_seconds / (len(dev_df) + len(llm_test_df))

print(f"\nZero-shot on {len(llm_test_df)} test tweets ({llm_latency_ms:.0f} ms per tweet):", quality(llm_test_df.label, llm_pred))
other = test_mean if RULE == "sum" else test_sum
print(f"(For information only, not used to choose: the other rule gives {quality(llm_test_df.label, other.argmax(1))})")
print("\nPredicted label counts:", dict(zip(VERBALIZERS, np.bincount(llm_pred, minlength=3))))
print("True label counts     :", dict(zip(VERBALIZERS, np.bincount(llm_test_df.label, minlength=3))))

In [ ]:
view = llm_test_df.assign(true=[LABELS[y] for y in llm_test_df.label], predicted=[LABELS[y] for y in llm_pred],
                          confidence=llm_probs.max(1).round(3))
pd.set_option("display.max_colwidth", 90)
view[["text", "true", "predicted", "confidence"]].head(8)

### Why not just generate the answer?
For contrast, here is what the same model writes when we let it **generate** freely. Scoring avoids parsing problems: it always returns one of our three labels plus a score for each.

In [ ]:
for tweet in llm_test_df.text.head(3):
    enc = tokenizer.apply_chat_template(build_messages(tweet), add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True).to(DEVICE)
    out = llm.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"{answer!r:>14}  <-  {tweet[:80]}")

✋ **Checkpoint (chat):** compare the predicted and true label counts. Does the LLM favour some label regardless of the tweet? How does its macro-F1 compare with the supervised models that saw 9,543 labels?

🧪 **Try it — contextual calibration (Zhao et al., 2021):** score a content-free input such as `"N/A"`, subtract its scores from every tweet's scores and re-evaluate. This removes part of the model's label prior bias.

<details><summary>Show a solution</summary>

```python
cf_sum, cf_mean = score_texts(["N/A"])
cf = cf_sum if RULE == "sum" else cf_mean
calibrated_pred = (llm_scores - cf).argmax(1)
print(quality(llm_test_df.label, calibrated_pred))
```
Strictly, you should also decide *whether* to calibrate on the development set, not on the test set.
</details>

## Part 4 — Synthetic data augmentation for a rare class

Bearish is the rarest class and, for Andes Bank, the most valuable. Suppose we only had **120 labelled tweets** (about 18 bearish). Can the LLM write extra bearish examples that help a cheap classifier?

Recipe: seed the prompt with a few **real training** examples → generate at temperature 0.9 → clean and deduplicate → read them → train on real + synthetic → **evaluate on real test data only** and compare with the real-only model.

In [ ]:
MINORITY = 0                                             # 0 = bearish
small_df = stratified_sample(train_df, N_SMALL, seed=SEED)
print("Small training set:", dict(zip(LABELS, np.bincount(small_df.label, minlength=3))))

def train_eval(train_texts, train_labels, name):
    """TF-IDF + logistic regression trained on the given texts, evaluated on the real test set."""
    vec = make_vectorizer()
    clf = make_lr().fit(vec.fit_transform(train_texts), train_labels)
    pred = clf.predict(vec.transform(test_df.text))
    rep = classification_report(test_df.label, pred, output_dict=True, zero_division=0)
    row = {"model": name, **quality(test_df.label, pred),
           "minority_precision": round(rep[str(MINORITY)]["precision"], 4),
           "minority_recall": round(rep[str(MINORITY)]["recall"], 4),
           "minority_f1": round(rep[str(MINORITY)]["f1-score"], 4)}
    return row, vec, clf

row_real, vec_small, lr_small = train_eval(small_df.text, small_df.label, "LR · 120 real")
pd.DataFrame([row_real])

In [ ]:
import re

def clean(text):
    """First non-empty line, without quotes, bullets or a 'Tweet:' prefix."""
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    if not lines:
        return ""
    t = re.sub(r"^(tweet\s*:|[-*•]\s*|\d+[.)]\s*)", "", lines[0], flags=re.I).strip()
    return t.strip('"“”\' ')

def generate_examples(label_name, pool, n, temperature=0.9, per_call=10, seed=SEED):
    """Ask the LLM for n new tweets of one class, few-shot seeded with 3 real training examples per call."""
    rs = np.random.RandomState(seed)
    torch.manual_seed(seed)
    outputs = []
    while len(outputs) < n:
        shots = rs.choice(pool, size=min(3, len(pool)), replace=False)
        examples = "\n".join(f"- {s}" for s in shots)
        messages = [{"role": "system", "content": "You write realistic, short financial news tweets."},
                    {"role": "user", "content": f"Here are examples of {label_name} financial news tweets:\n"
                                                f"{examples}\n\nWrite one new {label_name} financial news tweet "
                                                "about a different company. Output only the tweet."}]
        enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                            return_tensors="pt", return_dict=True).to(DEVICE)
        k = min(per_call, n - len(outputs))
        out = llm.generate(**enc, do_sample=True, temperature=temperature, top_p=0.95,
                           max_new_tokens=GEN_MAX_NEW_TOKENS, num_return_sequences=k,
                           pad_token_id=tokenizer.eos_token_id)
        outputs += tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outputs

pool = small_df[small_df.label == MINORITY].text.values
t0 = time.time()
generated = generate_examples(LABELS[MINORITY], pool, N_SYNTH)
print(f"Generated {len(generated)} texts in {time.time() - t0:.1f} s")

# Filter: clean, drop very short texts, exact duplicates and copies of real training tweets
seen = {t.lower() for t in train_df.text}
synthetic = []
for g in generated:
    t = clean(g)
    if len(t.split()) >= 4 and t.lower() not in seen:
        synthetic.append(t)
        seen.add(t.lower())
print(f"Kept {len(synthetic)} of {len(generated)} after cleaning and deduplication\n")
for t in synthetic[:10]:
    print("  •", t)

**Read them.** Are they really bearish? Do they repeat the same companies or templates? A small model can drift off-label (a "bearish" request that returns good news) — that is label noise you would be adding to your training set.

A quick, rough **diversity check**: the average cosine similarity between pairs of synthetic tweets vs pairs of real bearish tweets (higher = more repetitive).

In [ ]:
def mean_pairwise_cosine(texts):
    if len(texts) < 2:
        return float("nan")
    E = encoder.encode(list(texts), convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    S = E @ E.T
    return float(S[np.triu_indices(len(texts), k=1)].mean())

real_minority = train_df[train_df.label == MINORITY].text.sample(n=max(2, len(synthetic)), random_state=SEED)
print(f"Mean pairwise similarity · synthetic: {mean_pairwise_cosine(synthetic):.3f} | real {LABELS[MINORITY]}: {mean_pairwise_cosine(real_minority):.3f}")

In [ ]:
# Synthetic examples go into the TRAINING set only. The test set stays 100% real.
aug_texts = list(small_df.text) + synthetic
aug_labels = list(small_df.label) + [MINORITY] * len(synthetic)
row_aug, vec_aug, lr_aug = train_eval(aug_texts, aug_labels, f"LR · 120 real + {len(synthetic)} synthetic")
aug_table = pd.DataFrame([row_real, row_aug]).set_index("model")
aug_table

✋ **Checkpoint (chat):** did the bearish recall go up? What happened to bearish precision and to macro-F1? Would you ship the augmented model?

🧪 **Try it:** generate at temperature 0.5 and at 1.2. Which gives more diverse examples, and which more off-label ones?

<details><summary>Show a solution</summary>

```python
for temp in (0.5, 1.2):
    texts = [clean(g) for g in generate_examples(LABELS[MINORITY], pool, 10, temperature=temp)]
    print(temp, round(mean_pairwise_cosine(texts), 3), texts[:3])
```
Lower temperature → more repetitive; higher → more diverse but more off-topic or off-label text. Read them either way.
</details>

## Part 5 — Summary: quality, calibration, latency and cost

We evaluate every model on the **same** tweets (the LLM test subset), so the comparison is fair. For each model we report:
- **accuracy** and **macro-F1**;
- **ECE** (expected calibration error): the average gap between confidence and accuracy over 10 confidence bins (0 = perfectly calibrated). With few tweets it is noisy — read it as a rough signal;
- **latency**: milliseconds per tweet on *this* machine, processing the whole subset;
- **estimated cost per 1,000 predictions** if you run the model yourself = latency × price of the machine per hour.

⚠️ The prices below are **placeholders**, not quotes. Replace them with current prices from your provider and write down the date.

In [ ]:
COMPUTE_USD_PER_HOUR = 1.00      # placeholder: price per hour of the machine that runs the model
API_USD_PER_M_TOKENS = 1.00      # placeholder: price per million tokens of an LLM API

def ece(y_true, probs, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1)
    correct = (pred == np.asarray(y_true)).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = (conf > lo) & (conf <= hi)
        if in_bin.any():
            total += in_bin.mean() * abs(correct[in_bin].mean() - conf[in_bin].mean())
    return total

texts, y_sub = llm_test_df.text.tolist(), llm_test_df.label.values

def timed(fn):
    t0 = time.perf_counter()
    probs = fn()
    return probs, 1000 * (time.perf_counter() - t0) / len(texts)

def mlp_probs():
    with torch.no_grad():
        return torch.softmax(mlp(to_tensor(embed(texts)).to(DEVICE)), 1).cpu().numpy()

candidates = {
    "Naive Bayes · TF-IDF": (len(train_df), lambda: nb.predict_proba(vectorizer.transform(texts))),
    "Logistic regression · TF-IDF": (len(train_df), lambda: lr.predict_proba(vectorizer.transform(texts))),
    "PyTorch MLP · MiniLM": (len(train_df), mlp_probs),
    "LR · 120 real": (N_SMALL, lambda: lr_small.predict_proba(vec_small.transform(texts))),
    f"LR · 120 real + {len(synthetic)} synthetic": (N_SMALL, lambda: lr_aug.predict_proba(vec_aug.transform(texts))),
}
summary = []
for name, (n_labels, fn) in candidates.items():
    probs, ms = timed(fn)
    summary.append({"model": name, "labels used": n_labels, **quality(y_sub, probs.argmax(1)),
                    "ECE": round(ece(y_sub, probs), 4), "ms per tweet": round(ms, 2)})
summary.append({"model": f"Qwen2.5-0.5B zero-shot ({RULE})", "labels used": 0, **quality(y_sub, llm_pred),
                "ECE": round(ece(y_sub, llm_probs), 4), "ms per tweet": round(llm_latency_ms, 2)})
summary = pd.DataFrame(summary).set_index("model")
summary["USD per 1,000 (self-hosted)"] = (summary["ms per tweet"] / 1000 * 1000 / 3600 * COMPUTE_USD_PER_HOUR).round(4)
print(f"Evaluated on the same {len(texts)} test tweets | device: {DEVICE}")
summary

In [ ]:
avg_prompt_tokens = np.mean([len(prompt_ids(t)) for t in texts]) + 2   # prompt + a short answer
print(f"An API LLM would read about {avg_prompt_tokens:.0f} tokens per tweet: "
      f"≈ USD {avg_prompt_tokens * 1000 / 1e6 * API_USD_PER_M_TOKENS:.4f} per 1,000 tweets at the placeholder price.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(summary["ms per tweet"], summary["macro_f1"])
for name, r in summary.iterrows():
    ax.annotate(name, (r["ms per tweet"], r["macro_f1"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xscale("log"); ax.set_xlabel("ms per tweet (log scale)"); ax.set_ylabel("macro-F1")
ax.set_title("Quality vs latency on the same test tweets"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

✋ **Checkpoint (chat):** Andes Bank must classify 50,000 tweets a day and cannot send data outside. Which model would you deploy, and which one would you use only to bootstrap labels? Use two numbers from the table.

## Wrap-up
- **Generative vs discriminative:** Naive Bayes learned faster with little data; logistic regression caught up and passed it with more (look at your learning curve).
- **Pretrained representations** (sentence embeddings, VAE latents) let a small PyTorch MLP benefit from knowledge learned without our labels.
- **An LLM is a generative classifier:** scoring the log-likelihood of each label gives valid labels and scores for every class, with zero training labels — but it is slow, biased towards some labels, and must be validated on real labelled data.
- **Synthetic data** can raise the recall of a rare class; it goes into training only, and it must be read and filtered.
- **Decide with several columns:** macro-F1, calibration, latency, cost and privacy.

**Next:** open the **challenge** notebook. You will repeat these experiments with your own training size, minority class, test slice and business constraint.